
## 구글 드라이브 연결(새로 열 때마다 실행)


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## 음성 ZIP을 Colab에 풀기

In [3]:
import shutil, os, glob, unicodedata, zipfile
BASE = '/content/drive/MyDrive/emotion_ser'
NAMES = ['5차년도', '5차년도_2차']
def find(name, ext):
    for p in glob.glob(BASE + '/**/*.' + ext, recursive=True):
        if unicodedata.normalize('NFC', os.path.basename(p)) == name + '.' + ext:
            return p

for n in NAMES:
    out = '/content/wav_' + n
    if not os.path.isdir(out):
        shutil.copy(find(n, 'zip'), '/content/tmp.zip')
        zipfile.ZipFile('/content/tmp.zip').extractall(out)
        os.remove('/content/tmp.zip')
    print(n, len(glob.glob(out + '/**/*.wav', recursive=True)))

5차년도 10011
5차년도_2차 19374


## CSV와 음성 연결, 감정 5개로 정리

In [4]:
import pandas as pd
from collections import Counter

frames = []
for n in NAMES:
    df = pd.read_csv(find(n, 'csv'), encoding='cp949')
    emo = [c for c in df.columns if c.endswith('감정')]
    res = df[emo].astype(str).apply(lambda r: Counter(x.strip().lower() for x in r).most_common(1)[0], axis=1)
    df['label'] = res.str[0]
    df['votes'] = res.str[1]
    wavs = {os.path.basename(p)[:-4]: p for p in glob.glob('/content/wav_' + n + '/**/*.wav', recursive=True)}
    df['path'] = df['wav_id'].map(wavs)
    df['src'] = n
    print(n, '연결:', df['path'].notna().sum(), '/', len(df))
    frames.append(df)
all_df = pd.concat(frames, ignore_index=True)

mp = {'happiness':'happy','neutral':'neutral','sadness':'sad','angry':'angry','disgust':'angry','fear':'anxious'}
d = all_df[(all_df['votes'] >= 3) & all_df['path'].notna() & all_df['label'].isin(mp)].copy()
d['y'] = d['label'].map(mp)
print(len(d))
print(d['y'].value_counts())

5차년도 연결: 10011 / 10011
5차년도_2차 연결: 19374 / 19374
22447
y
sad        8649
angry      4425
neutral    4308
happy      3409
anxious    1656
Name: count, dtype: int64


## MFCC 특징 추출, 감정별 1000개 뽑기

In [5]:
import numpy as np, librosa
from joblib import Parallel, delayed

N_PER = 1000
CACHE = BASE + '/feat_' + '_'.join(NAMES) + '_' + str(N_PER) + '.npz'

def feat(path):
    y, sr = librosa.load(path, sr=16000, duration=6)
    m = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    return np.concatenate([m.mean(axis=1), m.std(axis=1)])

sub = pd.concat([g.sample(min(len(g), N_PER), random_state=0) for _, g in d.groupby('y')]).reset_index(drop=True)
if os.path.exists(CACHE):
    z = np.load(CACHE, allow_pickle=True)
    X, Y = z['X'], z['Y']
    print('저장본 불러옴')
else:
    X = np.array(Parallel(n_jobs=-1)(delayed(feat)(p) for p in sub['path']))
    Y = sub['y'].values
    np.savez(CACHE, X=X, Y=Y)
print(X.shape, sub['y'].value_counts().to_dict())

저장본 불러옴
(5000, 80) {'angry': 1000, 'anxious': 1000, 'happy': 1000, 'neutral': 1000, 'sad': 1000}


## SVM 기준선(macro-F1 0.52)

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix, f1_score

Xtr, Xte, ytr, yte = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=0)
clf = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=10, class_weight='balanced'))
clf.fit(Xtr, ytr)
pred = clf.predict(Xte)
print(classification_report(yte, pred))
print('macro-F1:', f1_score(yte, pred, average='macro'))
labels = sorted(set(Y))
print(labels)
print(confusion_matrix(yte, pred, labels=labels))

              precision    recall  f1-score   support

       angry       0.45      0.49      0.47       200
     anxious       0.47      0.41      0.44       200
       happy       0.52      0.49      0.51       200
     neutral       0.67      0.69      0.68       200
         sad       0.51      0.53      0.52       200

    accuracy                           0.52      1000
   macro avg       0.52      0.52      0.52      1000
weighted avg       0.52      0.52      0.52      1000

macro-F1: 0.5228222042719952
['angry', 'anxious', 'happy', 'neutral', 'sad']
[[ 99  31  30  12  28]
 [ 37  82  23  15  43]
 [ 33  19  99  31  18]
 [ 18  12  20 139  11]
 [ 35  31  18  11 105]]


## 분할 저장(한 번 만들면 그대로 사용)

In [6]:
from sklearn.model_selection import train_test_split
SPLIT = BASE + '/split_v1.csv'
if os.path.exists(SPLIT):
    sp = pd.read_csv(SPLIT)
    print('저장된 분할 불러옴')
else:
    s = sub.copy()
    tr0, te0 = train_test_split(s, test_size=0.2, stratify=s['y'], random_state=0)
    tr0, va0 = train_test_split(tr0, test_size=0.125, stratify=tr0['y'], random_state=0)
    s['split'] = 'train'
    s.loc[va0.index, 'split'] = 'val'
    s.loc[te0.index, 'split'] = 'test'
    sp = s[['wav_id', '발화문', 'y', 'split']].copy()
    sp.to_csv(SPLIT, index=False, encoding='utf-8-sig')
print(sp['split'].value_counts())

저장된 분할 불러옴
split
train    3500
test     1000
val       500
Name: count, dtype: int64


## 음성 불러오기와 모델 준비

In [10]:
# 셀 7: 음성 불러오기와 wav2vec 모델 준비
import torch, numpy as np, librosa
from joblib import Parallel, delayed
from torch.utils.data import Dataset
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

print('GPU:', torch.cuda.is_available())
pm = all_df.drop_duplicates('wav_id').set_index('wav_id')['path']
sp['path'] = sp['wav_id'].map(pm)
print('경로 연결:', sp['path'].notna().sum(), '/', len(sp))

def load(p):
    w, _ = librosa.load(p, sr=16000, duration=6)
    return w.astype(np.float32)
sp['wav'] = Parallel(n_jobs=-1)(delayed(load)(p) for p in sp['path'])

MODEL = 'facebook/wav2vec2-xls-r-300m'
labels = sorted(sp['y'].unique())
l2i = {l: i for i, l in enumerate(labels)}
fe = AutoFeatureExtractor.from_pretrained(MODEL)

class DS(Dataset):
    def __init__(self, wavs, ys):
        self.w, self.y = list(wavs), list(ys)
    def __len__(self):
        return len(self.w)
    def __getitem__(self, i):
        return {'w': self.w[i], 'label': l2i[self.y[i]]}

def collate(batch):
    out = fe([b['w'] for b in batch], sampling_rate=16000, padding=True, return_tensors='pt', return_attention_mask=True)
    out['labels'] = torch.tensor([b['label'] for b in batch])
    return out

tr = sp[sp['split'] == 'train']
va = sp[sp['split'] == 'val']
model = AutoModelForAudioClassification.from_pretrained(MODEL, num_labels=len(labels), label2id=l2i, id2label={i: l for l, i in l2i.items()})
model.freeze_feature_encoder()

def metrics(p):
    pred = p.predictions.argmax(-1)
    return {'macro_f1': f1_score(p.label_ids, pred, average='macro'), 'acc': accuracy_score(p.label_ids, pred)}

args = TrainingArguments(output_dir='/content/ckpt', per_device_train_batch_size=8, per_device_eval_batch_size=8, gradient_accumulation_steps=2, learning_rate=3e-5, warmup_steps=260, num_train_epochs=12, eval_strategy='epoch', save_strategy='epoch', save_total_limit=1, load_best_model_at_end=True, metric_for_best_model='macro_f1', fp16=True, logging_steps=50, remove_unused_columns=False, dataloader_num_workers=2, report_to='none')
trainer = Trainer(model=model, args=args, train_dataset=DS(tr['wav'], tr['y']), eval_dataset=DS(va['wav'], va['y']), data_collator=collate, compute_metrics=metrics)

GPU: True
경로 연결: 5000 / 5000


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.weight | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
projector.weight             | MISSING    | 
classifier.weight            | MISSING    | 
projector.bias               | MISSING    | 
classifier.bias              | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 학습

In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1,Acc
1,3.215916,1.600375,0.165145,0.268000
2,3.072876,1.468725,0.225129,0.322000
3,2.832430,1.299754,0.373030,0.428000
4,2.528073,1.205773,0.412054,0.464000
5,2.418911,1.150886,0.526839,0.542000
6,2.249780,1.119506,0.520717,0.530000
7,2.147884,1.081839,0.553640,0.560000
8,1.990833,1.025388,0.585919,0.594000
9,1.707283,1.002193,0.600369,0.604000
10,1.670385,0.982900,0.637994,0.640000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (4 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2628, training_loss=2.258428689733125, metrics={'train_runtime': 6552.4124, 'train_samples_per_second': 6.41, 'train_steps_per_second': 0.401, 'total_flos': 7.63318486828368e+18, 'train_loss': 2.258428689733125, 'epoch': 12.0})

## 시험 평가, 3분류 계산, 확률 저장, 모델 저장

In [12]:
from scipy.special import softmax
rows = []
for name in ['val', 'test']:
    s = sp[sp['split'] == name]
    pr = softmax(trainer.predict(DS(s['wav'], s['y'])).predictions, axis=1)
    r = s[['wav_id', 'y', 'split']].copy()
    for i, l in enumerate(labels):
        r['p_' + l] = pr[:, i]
    rows.append(r)
res = pd.concat(rows)
res.to_csv(BASE + '/audio_probs_v2.csv', index=False, encoding='utf-8-sig')

t = res[res['split'] == 'test']
pred = np.array(labels)[t[['p_' + l for l in labels]].values.argmax(1)]
print(classification_report(t['y'], pred))
print('5분류 macro-F1:', f1_score(t['y'], pred, average='macro'))
print(labels)
print(confusion_matrix(t['y'], pred, labels=labels))

to3 = {'happy': 'good', 'neutral': 'flat', 'sad': 'bad', 'angry': 'bad', 'anxious': 'bad'}
y3 = t['y'].map(to3)
p3 = pd.Series(pred, index=t.index).map(to3)
print('3분류 macro-F1:', f1_score(y3, p3, average='macro'))

model.half().save_pretrained(BASE + '/model_w2v2_v2')
fe.save_pretrained(BASE + '/model_w2v2_v2')

              precision    recall  f1-score   support

       angry       0.52      0.52      0.52       200
     anxious       0.72      0.60      0.65       200
       happy       0.76      0.76      0.76       200
     neutral       0.73      0.72      0.73       200
         sad       0.57      0.69      0.63       200

    accuracy                           0.66      1000
   macro avg       0.66      0.66      0.66      1000
weighted avg       0.66      0.66      0.66      1000

5분류 macro-F1: 0.6575978219654366
['angry', 'anxious', 'happy', 'neutral', 'sad']
[[103  27  20  11  39]
 [ 33 120   3   4  40]
 [ 24   0 152  19   5]
 [ 14   3  21 144  18]
 [ 24  17   3  18 138]]
3분류 macro-F1: 0.7957009334602696


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['/content/drive/MyDrive/emotion_ser/model_w2v2_v2/preprocessor_config.json']